# Bidirectional Recurrent Neural Network (BiRNN)

A Bidirectional Recurrent Neural Network (BiRNN) is a neural network that processes sequence data in **both directions**.

- **Forward direction** → from past → future  
- **Backward direction** → from future → past  

This helps the model understand context from both sides of a sequence.

---

# Why Normal RNN is Limited

A standard RNN reads data only in one direction.

### Example Sentence

```text
"I went to the bank to deposit money"
```

A normal RNN reads:

```text
Left → Right
```

When it sees the word **"bank"**, it only knows previous words:

```text
I went to the bank
```

It does not yet know:

```text
deposit money
```

which gives the real meaning of **bank**  
(financial institution, not river bank).

---

# How Bidirectional Network Solves This

A bidirectional network uses **two RNNs**:

## Forward RNN

Reads:

```text
I → went → to → the → bank → to → deposit → money
```

## Backward RNN

Reads:

```text
money → deposit → to → bank → the → to → went → I
```

At each word, outputs from both networks are combined.


For the Word "bank"

- Forward network knows the **past context**
- Backward network knows the **future context**

So the model understands meaning better.

---

# Architecture

A bidirectional network has:

```text
        Input Sequence
               ↓

     ┌─────────────────┐
     │   Forward RNN   │ → Hidden State
     │    / LSTM       │
     └─────────────────┘

     ┌─────────────────┐
     │  Backward RNN   │ → Hidden State
     │    / LSTM       │
     └─────────────────┘

               ↓

     Concatenate Outputs

               ↓

       Final Prediction
```

---

# Mathematical Representation

## Forward Hidden State

$$
ht​
​=f(xt​,ht−1​)
$$


## Backward Hidden State

$$
ht​
​=f(xt​,ht+1​)
$$


## Final Output

$$
ht	​=[ht;ht]
$$

---

# Types

A bidirectional network can be built with:

- Bidirectional RNN (BiRNN)
- Bidirectional LSTM (BiLSTM) — most common
- Bidirectional GRU (BiGRU)

BiLSTM is widely used because LSTM handles long-term dependencies better.

---

# Example

### Sentence

```text
"The food was not good"
```

If the model reads only:

```text
Left → Right
```

At the word **"good"**, it may think sentiment is positive.

A bidirectional model sees:

```text
not good
```

from future/past context and predicts negative sentiment correctly.

---

# Applications

Used in:

- Natural Language Processing (NLP)
- Time-Series Analysis
- Video Understanding
- Medical Signal Analysis

---

# Advantages

- Better contextual understanding
- Higher accuracy in sequence tasks
- Uses both past and future information

---

# Disadvantages

- More computational cost
- Slower training
- Cannot be used easily in real-time prediction because it needs future information

In [1]:
import pandas as pd
import numpy as np

In [19]:
df=pd.read_csv('/content/drive/MyDrive/AI Training/Datasets/SMSSpamCollection.csv')

In [22]:
df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [21]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
df['label']=le.fit_transform(df['label'])

In [23]:
df['label'].value_counts()

,count
label,
0,4825
1,747


In [24]:


from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test=train_test_split(df['message'],df['label'])

In [25]:
x_test.shape

(1393,)

In [27]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding,LSTM,Dense,Dropout,Bidirectional)

In [28]:
vocab_size = 5000

tokenizer = Tokenizer(num_words=vocab_size)

tokenizer.fit_on_texts(x_train)

X_train_seq = tokenizer.texts_to_sequences(x_train)
X_test_seq = tokenizer.texts_to_sequences(x_test)

In [29]:
max_length = 100

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding="post"
)

In [30]:
print(X_train_pad.shape)

(4179, 100)


In [31]:
model = Sequential([

    Embedding(input_dim=vocab_size,output_dim=128,input_length=max_length),
    Bidirectional(LSTM(64,return_sequences=False)),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [32]:

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [33]:
history = model.fit(X_train_pad,y_train,epochs=5,validation_split=0.2)

Epoch 1/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 22s 170ms/step - accuracy: 0.9195 - loss: 0.2232 - val_accuracy: 0.9880 - val_loss: 0.0505
Epoch 2/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 19s 155ms/step - accuracy: 0.9904 - loss: 0.0391 - val_accuracy: 0.9904 - val_loss: 0.0355
Epoch 3/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 20s 153ms/step - accuracy: 0.9961 - loss: 0.0163 - val_accuracy: 0.9916 - val_loss: 0.0371
Epoch 4/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 16s 152ms/step - accuracy: 0.9976 - loss: 0.0092 - val_accuracy: 0.9904 - val_loss: 0.0385
Epoch 5/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 16s 152ms/step - accuracy: 0.9991 - loss: 0.0039 - val_accuracy: 0.9928 - val_loss: 0.0450


In [34]:
loss, accuracy = model.evaluate(
    X_test_pad,
    y_test
)

44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.9842 - loss: 0.0763


In [35]:
print(accuracy)

0.9842067360877991


In [37]:
def predict_spam(text):

    sequence = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding="post"
    )

    prediction = model.predict(padded)[0][0]

    if prediction > 0.5:
        print("SPAM")
    else:
        print("NOT SPAM")

    print("Confidence:", prediction)


predict_spam("Congratulations! You won a free iPhone")

predict_spam("Are you coming to class today?")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
SPAM
Confidence: 0.9568156
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step
NOT SPAM
Confidence: 0.0012849089
